# BraTS 2026 Task 5 – FlowLet3D Brain Tumor Inpainting

This notebook trains and evaluates the **FlowLet3D_Inpainting** model for healthy-tissue reconstruction inside brain tumor regions.

## Architecture summary
| Component | Description |
|---|---|
| **DWT3D / IDWT3D** | 3-D Haar wavelet decomposition / reconstruction (8 sub-bands) |
| **FlowLet3DBlock** | Conditional velocity UNet: FiLM + cross-attention |
| **ConditionalFlowMatching3D** | Rectified / CFM / trigonometric / VP-diffusion flow matching |
| **SymmetryAttention3D** | Contralateral hemisphere guidance |
| **GlobalContextBranch** | Full-volume anatomy encoder |
| **LocalMaskBranch** | Tumor-region context encoder |
| **ContextFusion** | Cross-attention fusion of global & local context |
| **Encoder3D / Decoder3D** | Multi-scale 3-D U-Net backbone with skip connections |

In [ ]:
# ── 1. Environment setup ──────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('monai[nibabel,tqdm]>=1.3', 'pywavelets', 'nibabel', 'einops')

# Optional: xformers for memory-efficient attention
try:
    import xformers
    print('xformers already available')
except ImportError:
    print('xformers not installed – using PyTorch native attention (safe to ignore)')

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────
import os, math, time, copy, logging
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

# Add repo to path when running from Kaggle
import sys
sys.path.insert(0, '/kaggle/working')

from config  import Config
from dataset import build_datasets
from model   import FlowLet3D_Inpainting
from losses  import InpaintingLoss
from validate import validate, print_report
from train   import EMA, _save, _load, _warmup_cosine_schedule

from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

logging.basicConfig(level=logging.INFO)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────
cfg = Config(
    data_root       = '/kaggle/input/brats-inpainting-training',
    checkpoint_dir  = '/kaggle/working/checkpoints',
    log_dir         = '/kaggle/working/logs',

    # Model
    model_channels  = 64,
    channel_mult    = (1, 2, 4, 8),
    num_res_blocks  = 2,
    num_heads       = 8,
    context_dim     = 128,
    flow_type       = 'rectified',   # rectified | cfm | trigonometric | vp_diffusion
    num_flow_steps  = 50,

    # Patch size – use 128^3 if >= 24 GB VRAM
    patch_size      = (96, 96, 96),

    # Training
    batch_size      = 1,
    num_epochs      = 200,
    learning_rate   = 1e-4,
    warmup_epochs   = 10,
    amp             = True,
    use_checkpoint  = True,
    ema_decay       = 0.9999,

    # Losses
    lambda_mask     = 1.0,
    lambda_global   = 0.5,
    lambda_ssim     = 0.5,
    lambda_flow     = 0.2,
    lambda_fft      = 0.1,
    lambda_sym      = 0.1,
    lambda_edge     = 0.1,

    # Inference
    sw_batch_size   = 2,
    sw_overlap      = 0.5,
    tta_flips       = [2, 3, 4],
)

print('Config:', cfg)

In [ ]:
# ── 4. Data loaders ───────────────────────────────────────────────────────
import logging
log = logging.getLogger('data')
train_loader, val_loader = build_datasets(cfg, logger=log)
print(f'Train batches: {len(train_loader)}   Val batches: {len(val_loader)}')

# Preview one batch
sample = next(iter(train_loader))
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: {tuple(v.shape)}  range=[{v.min():.2f}, {v.max():.2f}]')

In [ ]:
# ── 5. Build model ────────────────────────────────────────────────────────
model = FlowLet3D_Inpainting.from_config(cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params/1e6:.1f} M')

# Smoke test with a tiny forward pass
with torch.no_grad():
    dummy_v = torch.randn(1, 1, 32, 32, 32, device=DEVICE)
    dummy_m = torch.zeros(1, 1, 32, 32, 32, device=DEVICE)
    dummy_m[:, :, 8:24, 8:24, 8:24] = 1.0
    out = model(dummy_v, dummy_m)
    print(f'Smoke test pred shape: {out["pred"].shape}')
    print('Smoke test PASSED ✓')

In [ ]:
# ── 6. Loss, optimiser, scheduler ─────────────────────────────────────────
criterion = InpaintingLoss.from_config(cfg).to(DEVICE)

optim = AdamW(model.parameters(),
              lr=cfg.learning_rate, weight_decay=cfg.weight_decay)

total_steps  = cfg.num_epochs * len(train_loader)
warmup_steps = cfg.warmup_epochs * len(train_loader)

sched = LambdaLR(
    optim,
    lr_lambda=lambda s: _warmup_cosine_schedule(s, warmup_steps, total_steps)
)

scaler = GradScaler(enabled=cfg.amp)
ema    = EMA(model, decay=cfg.ema_decay)

print('Optimiser and scheduler ready.')

In [ ]:
# ── 7. Optional: resume from checkpoint ───────────────────────────────────
RESUME_PATH  = None   # Set to e.g. '/kaggle/working/checkpoints/ckpt_best.pt'
start_epoch  = 0
best_psnr    = -float('inf')
no_improve   = 0

if RESUME_PATH and os.path.isfile(RESUME_PATH):
    start_epoch, prev_metrics = _load(RESUME_PATH, model, ema, optim, sched, DEVICE)
    best_psnr = prev_metrics.get('psnr_mask', -float('inf'))
    print(f'Resumed from epoch {start_epoch}  best PSNR_mask={best_psnr:.2f}')

In [ ]:
# ── 8. Training loop ──────────────────────────────────────────────────────
from IPython.display import clear_output

history = {'epoch': [], 'loss': [], 'psnr_mask': [], 'ssim_mask': []}

for epoch in range(start_epoch, cfg.num_epochs):
    model.train()
    t0         = time.time()
    epoch_loss = 0.0
    n_batches  = 0
    loss_parts = {k: 0.0 for k in
                  ['mask','global','ssim','flow','fft','sym','edge']}

    for batch in train_loader:
        voided = batch['voided'].to(DEVICE, non_blocking=True)
        mask   = batch['mask'].to(DEVICE,   non_blocking=True)
        target = batch['image'].to(DEVICE,  non_blocking=True)

        optim.zero_grad(set_to_none=True)

        with autocast(enabled=cfg.amp):
            out       = model(voided, mask)
            pred      = out['pred']
            flow_loss = model.compute_flow_loss(voided, target, mask)
            losses    = criterion(pred, target, mask, flow_loss)
            loss      = losses['total']

        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optim)
        scaler.update()
        sched.step()
        ema.update(model)

        epoch_loss += loss.item()
        for k in loss_parts:
            loss_parts[k] += losses.get(k, torch.tensor(0.)).item()
        n_batches += 1

    epoch_loss /= max(n_batches, 1)
    history['epoch'].append(epoch)
    history['loss'].append(epoch_loss)

    print(f'Epoch {epoch:04d}  loss={epoch_loss:.4f}  '
          f'lr={sched.get_last_lr()[0]:.2e}  '
          f'dt={time.time()-t0:.1f}s')

    # Save checkpoint every N epochs
    if (epoch + 1) % cfg.save_every == 0:
        _save(model, ema, optim, sched, epoch, {}, cfg, tag='last')

    # Validation
    if (epoch + 1) % cfg.validate_every == 0:
        ema.apply(model)
        metrics = validate(model, val_loader, DEVICE, cfg, epoch)
        ema.restore(model)
        print_report(metrics)

        history['psnr_mask'].append(metrics.get('psnr_mask', 0))
        history['ssim_mask'].append(metrics.get('ssim_mask', 0))

        psnr = metrics.get('psnr_mask', -float('inf'))
        if psnr > best_psnr:
            best_psnr  = psnr
            no_improve = 0
            _save(model, ema, optim, sched, epoch, metrics, cfg, tag='best')
            print(f'  ★ New best PSNR_mask: {best_psnr:.2f} dB')
        else:
            no_improve += 1
            if no_improve >= cfg.early_stop_patience:
                print('Early stopping triggered.')
                break

print('Training complete.')

In [ ]:
# ── 9. Training curves ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['epoch'], history['loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')

val_epochs = history['epoch'][cfg.validate_every-1::cfg.validate_every]
if history['psnr_mask']:
    axes[1].plot(val_epochs[:len(history['psnr_mask'])], history['psnr_mask'])
    axes[1].set_title('Val PSNR (mask)'); axes[1].set_xlabel('Epoch')

if history['ssim_mask']:
    axes[2].plot(val_epochs[:len(history['ssim_mask'])], history['ssim_mask'])
    axes[2].set_title('Val SSIM (mask)'); axes[2].set_xlabel('Epoch')

plt.tight_layout(); plt.show()

In [ ]:
# ── 10. Final validation with EMA weights ─────────────────────────────────
best_ckpt = '/kaggle/working/checkpoints/ckpt_best.pt'
if os.path.isfile(best_ckpt):
    _load(best_ckpt, model, ema, device=DEVICE)
    ema.apply(model)

final_metrics = validate(model, val_loader, DEVICE, cfg, epoch='final')
print_report(final_metrics)

In [ ]:
# ── 11. Inference on a single case ────────────────────────────────────────
from infer import run_inference

# Replace with actual paths
VOIDED_PATH = '/kaggle/input/brats-inpainting-training/BraTS-GLI-00000-000/BraTS-GLI-00000-000-t1n-voided.nii.gz'
MASK_PATH   = '/kaggle/input/brats-inpainting-training/BraTS-GLI-00000-000/BraTS-GLI-00000-000-mask.nii.gz'
OUTPUT_PATH = '/kaggle/working/predicted_t1n.nii.gz'
CKPT_PATH   = '/kaggle/working/checkpoints/ckpt_best.pt'

if os.path.isfile(VOIDED_PATH) and os.path.isfile(MASK_PATH):
    run_inference(
        voided_path = VOIDED_PATH,
        mask_path   = MASK_PATH,
        output_path = OUTPUT_PATH,
        ckpt_path   = CKPT_PATH,
        cfg         = cfg,
        tta         = True,
    )
    print(f'Saved: {OUTPUT_PATH}')
else:
    print('Input files not found – adjust VOIDED_PATH and MASK_PATH above.')

In [ ]:
# ── 12. Visualise prediction (axial slice) ────────────────────────────────
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

if os.path.isfile(OUTPUT_PATH) and os.path.isfile(VOIDED_PATH):
    pred_vol   = nib.load(OUTPUT_PATH).get_fdata()
    voided_vol = nib.load(VOIDED_PATH).get_fdata()
    mask_vol   = nib.load(MASK_PATH).get_fdata()

    # Find a slice with tumour
    z = np.argmax(mask_vol.sum(axis=(0,1)))

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(voided_vol[:,:,z].T, cmap='gray', origin='lower')
    axes[0].set_title('Voided Input')
    axes[1].imshow(pred_vol[:,:,z].T,   cmap='gray', origin='lower')
    axes[1].set_title('Predicted Healthy')
    axes[2].imshow(mask_vol[:,:,z].T,   cmap='hot',  origin='lower', alpha=0.6)
    axes[2].set_title('Tumor Mask')
    for ax in axes: ax.axis('off')
    plt.suptitle(f'Axial slice z={z}')
    plt.tight_layout()
    plt.savefig('/kaggle/working/prediction_vis.png', dpi=150)
    plt.show()

In [ ]:
# ── 13. Batch inference on test set ───────────────────────────────────────
from pathlib import Path

TEST_ROOT   = '/kaggle/input/brats-inpainting-test'   # adjust as needed
OUTPUT_ROOT = '/kaggle/working/predictions'
CKPT_PATH   = '/kaggle/working/checkpoints/ckpt_best.pt'

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

if os.path.isdir(TEST_ROOT):
    for subj_dir in sorted(Path(TEST_ROOT).iterdir()):
        if not subj_dir.is_dir(): continue
        sid          = subj_dir.name
        voided_path  = subj_dir / f'{sid}-t1n-voided.nii.gz'
        mask_path    = subj_dir / f'{sid}-mask.nii.gz'
        output_path  = os.path.join(OUTPUT_ROOT, f'{sid}-predicted-t1n.nii.gz')

        if voided_path.exists() and mask_path.exists():
            print(f'Processing {sid} ...')
            run_inference(
                voided_path = str(voided_path),
                mask_path   = str(mask_path),
                output_path = output_path,
                ckpt_path   = CKPT_PATH,
                cfg         = cfg,
                tta         = True,
            )
    print('Batch inference complete.')
else:
    print(f'Test root not found: {TEST_ROOT}')